## NFL 4th Down Decision Analysis
##### Phase 1: Data Cleaning and Initial Preparation


#### About Dataset

***Source*** : Kaggle - [NFL Play-by-Play Data](https://www.kaggle.com/datasets/maxhorowitz/nflplaybyplay2009to2018)

***Description*** : 

### Dataset Overview
The NFL Play-by-Play dataset (2009-2018) contains:
- **449,371 rows** representing individual plays
- **255 columns** with detailed play information
- Data types: 135 float, 18 integer, 102 object columns
- Each row represents a single play, with various features describing the play's context, outcome, and other relevant information.

### Project Goal: 
Estimate probability in 4th down game scenarios and recommend whether a team should go for it, kick a field goal, or punt on 4th down.


### Objectives
- Import necessary libraries and packages
- Load and examine the NFL play-by-play dataset
- Identify and retain relevant features for 4th down analysis
- Initial dataset cleaning and preparation for Preliminary Analysis 


In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import joblib
import xgboost as xgb

import pandas as pd

import os

csv_path = "../../data/raw/NFL Play by Play 2009-2018 (v5).csv"

df = pd.read_csv(
    csv_path,
    low_memory=False  # avoid DtypeWarning for mixed-type columns
)

### Why these features matter for 4th-down decisions

- **qtr** and **game_seconds_remaining**: Capture game context and urgency; late-game 4th downs are often more aggressive.
- **posteam_score**, **defteam_score**, **score_differential**: Reflect the current game situation (ahead, tied, behind), which heavily influences risk tolerance.
- **yardline_100** and **side_of_field**: Encode field position; being deep in opponent territory vs. backed up near your own end zone changes the value of going for it vs. punting.
- **ydstogo** and **down**: Core football context; 4th-and-1 is very different from 4th-and-15 in conversion probability and strategy.
- **field_goal_attempt**, **field_goal_result**, **kick_distance**: Describe field goal opportunities and difficulty, informing the field goal vs. go-for-it tradeoff.
- **punt_attempt**: Indicates when the team chose to punt, providing data for the punt branch of the decision.
- **fourth_down_converted** and **fourth_down_failed**: Capture actual go-for-it outcomes, which are essential for learning conversion probabilities and payoffs.
- **touchdown**, **td_team**, **td_prob**, **fg_prob**, **ep**, **epa**: Provide outcome quality and advanced metrics (expected points and win-related probabilities) used to compare choices (go, kick, punt).
- **home_timeouts_remaining** and **away_timeouts_remaining**: Affect clock management and late-game decision constraints.


In [2]:
# Relevant features for the 4th down analysis 

fourth_down_features = [
    'game_id',
    'play_id',
    'home_team',
    'away_team',
    'qtr',
    'game_seconds_remaining',
    'posteam_score',
    'defteam_score',
    'score_differential',
    'posteam',
    'defteam',
    'fourth_down_converted',
    'fourth_down_failed',
    'drive',
    'side_of_field',
    'yardline_100',
    'down', 
    'ydstogo',
    'play_type',
    'yards_gained',
    'touchdown',
    'td_team',
    'field_goal_attempt',
    'field_goal_result',
    'kick_distance',
    'punt_attempt',
    'incomplete_pass',
    'interception',
    'home_timeouts_remaining',
    'away_timeouts_remaining',
    'td_prob', 
    'fg_prob',
    'ep',
    'epa'

]

# Filter the DataFrame to include only 4th down plays and relavant features

fourth_down_scenarios = (
    df.loc[df['down'] == 4, fourth_down_features]
      .copy()
)

    



In [3]:
# clean column names 

fourth_down_scenarios.columns = fourth_down_scenarios.columns.str.strip()


In [4]:
# inspect the first few rows of the filtered DataFrame

fourth_down_scenarios.head(10).style.set_table_attributes(
    'style="display:block; max-height:300px; overflow-y:scroll"'
)


,game_id,play_id,home_team,away_team,qtr,game_seconds_remaining,posteam_score,defteam_score,score_differential,posteam,defteam,fourth_down_converted,fourth_down_failed,drive,side_of_field,yardline_100,down,ydstogo,play_type,yards_gained,touchdown,td_team,field_goal_attempt,field_goal_result,kick_distance,punt_attempt,incomplete_pass,interception,home_timeouts_remaining,away_timeouts_remaining,td_prob,fg_prob,ep,epa
4,2009091000,139,PIT,TEN,1,3507.000000,0.000000,0.000000,0.000000,PIT,TEN,0.000000,0.000000,1,PIT,56.000000,4.000000,8,punt,0.000000,0.000000,nan,0.000000,nan,54.000000,1.000000,0.000000,0.000000,3,3,0.208111,0.244603,-0.699436,2.097796
8,2009091000,228,PIT,TEN,1,3394.000000,0.000000,0.000000,0.000000,TEN,PIT,0.000000,0.000000,2,TEN,96.000000,4.000000,8,punt,0.000000,0.000000,nan,0.000000,nan,50.000000,1.000000,0.000000,0.000000,3,3,0.135991,0.023641,-3.393288,-0.021313
14,2009091000,365,PIT,TEN,1,3205.000000,0.000000,0.000000,0.000000,PIT,TEN,0.000000,0.000000,3,TEN,41.000000,4.000000,21,punt,0.000000,0.000000,nan,0.000000,nan,30.000000,1.000000,0.000000,0.000000,3,3,0.150286,0.501696,0.757343,-0.311925
20,2009091000,522,PIT,TEN,1,3108.000000,0.000000,0.000000,0.000000,TEN,PIT,0.000000,0.000000,4,PIT,19.000000,4.000000,7,field_goal,0.000000,0.000000,nan,1.000000,missed,37.000000,0.000000,0.000000,0.000000,3,3,0.028760,0.895016,2.487035,-3.632243
24,2009091000,603,PIT,TEN,1,3002.000000,0.000000,0.000000,0.000000,PIT,TEN,0.000000,0.000000,5,PIT,79.000000,4.000000,16,punt,0.000000,0.000000,nan,0.000000,nan,53.000000,1.000000,0.000000,0.000000,3,3,0.157581,0.077646,-2.604561,1.526354
41,2009091000,998,PIT,TEN,2,2593.000000,0.000000,0.000000,0.000000,TEN,PIT,0.000000,0.000000,8,PIT,44.000000,4.000000,22,punt,0.000000,0.000000,nan,0.000000,nan,39.000000,1.000000,0.000000,0.000000,3,3,0.152789,0.441612,0.562832,0.201258
52,2009091000,1279,PIT,TEN,2,2245.000000,0.000000,0.000000,0.000000,PIT,TEN,0.000000,0.000000,9,PIT,62.000000,4.000000,5,punt,0.000000,0.000000,nan,0.000000,nan,46.000000,1.000000,0.000000,0.000000,3,3,0.186843,0.135106,-0.571651,-0.495752
65,2009091000,1633,PIT,TEN,2,1942.000000,0.000000,0.000000,0.000000,TEN,PIT,0.000000,0.000000,10,PIT,13.000000,4.000000,6,field_goal,0.000000,0.000000,nan,1.000000,blocked,31.000000,0.000000,0.000000,0.000000,3,3,0.005948,0.950566,2.816983,-3.236532
100,2009091000,2443,PIT,TEN,3,1512.000000,7.000000,7.000000,0.000000,PIT,TEN,0.000000,0.000000,15,TEN,45.000000,4.000000,1,punt,0.000000,0.000000,nan,0.000000,nan,38.000000,1.000000,0.000000,0.000000,3,3,0.309024,0.295397,0.958681,0.096593
104,2009091000,2563,PIT,TEN,3,1389.000000,7.000000,7.000000,0.000000,TEN,PIT,0.000000,0.000000,16,TEN,96.000000,4.000000,11,punt,0.000000,0.000000,nan,0.000000,nan,50.000000,1.000000,0.000000,0.000000,3,3,0.129661,0.025046,-3.424585,0.691486


In [5]:
fourth_down_scenarios.info()


<class 'pandas.core.frame.DataFrame'>
Index: 39644 entries, 4 to 449368
Data columns (total 34 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   game_id                  39644 non-null  int64  
 1   play_id                  39644 non-null  int64  
 2   home_team                39644 non-null  object 
 3   away_team                39644 non-null  object 
 4   qtr                      39644 non-null  int64  
 5   game_seconds_remaining   39640 non-null  float64
 6   posteam_score            39525 non-null  float64
 7   defteam_score            39525 non-null  float64
 8   score_differential       39525 non-null  float64
 9   posteam                  39525 non-null  object 
 10  defteam                  39525 non-null  object 
 11  fourth_down_converted    39525 non-null  float64
 12  fourth_down_failed       39525 non-null  float64
 13  drive                    39644 non-null  int64  
 14  side_of_field            3

In [6]:
# limiit all integers to 2 decimal places

# identify numeric columns
num_cols = fourth_down_scenarios.select_dtypes(include='number').columns

# round them to 2 decimal places
fourth_down_scenarios[num_cols] = fourth_down_scenarios[num_cols].round(2)



In [7]:
fourth_down_scenarios[num_cols].head(10)


,game_id,play_id,qtr,game_seconds_remaining,posteam_score,defteam_score,score_differential,fourth_down_converted,fourth_down_failed,drive,...,kick_distance,punt_attempt,incomplete_pass,interception,home_timeouts_remaining,away_timeouts_remaining,td_prob,fg_prob,ep,epa
4,2009091000,139,1,3507.0,0.0,0.0,0.0,0.0,0.0,1,...,54.0,1.0,0.0,0.0,3,3,0.21,0.24,-0.70,2.10
8,2009091000,228,1,3394.0,0.0,0.0,0.0,0.0,0.0,2,...,50.0,1.0,0.0,0.0,3,3,0.14,0.02,-3.39,-0.02
14,2009091000,365,1,3205.0,0.0,0.0,0.0,0.0,0.0,3,...,30.0,1.0,0.0,0.0,3,3,0.15,0.50,0.76,-0.31
20,2009091000,522,1,3108.0,0.0,0.0,0.0,0.0,0.0,4,...,37.0,0.0,0.0,0.0,3,3,0.03,0.90,2.49,-3.63
24,2009091000,603,1,3002.0,0.0,0.0,0.0,0.0,0.0,5,...,53.0,1.0,0.0,0.0,3,3,0.16,0.08,-2.60,1.53
41,2009091000,998,2,2593.0,0.0,0.0,0.0,0.0,0.0,8,...,39.0,1.0,0.0,0.0,3,3,0.15,0.44,0.56,0.20
52,2009091000,1279,2,2245.0,0.0,0.0,0.0,0.0,0.0,9,...,46.0,1.0,0.0,0.0,3,3,0.19,0.14,-0.57,-0.50
65,2009091000,1633,2,1942.0,0.0,0.0,0.0,0.0,0.0,10,...,31.0,0.0,0.0,0.0,3,3,0.01,0.95,2.82,-3.24
100,2009091000,2443,3,1512.0,7.0,7.0,0.0,0.0,0.0,15,...,38.0,1.0,0.0,0.0,3,3,0.31,0.30,0.96,0.10
104,2009091000,2563,3,1389.0,7.0,7.0,0.0,0.0,0.0,16,...,50.0,1.0,0.0,0.0,3,3,0.13,0.03,-3.42,0.69


In [8]:
# check for any null values 

fourth_down_scenarios.isnull().sum()

game_id                        0
play_id                        0
home_team                      0
away_team                      0
qtr                            0
game_seconds_remaining         4
posteam_score                119
defteam_score                119
score_differential           119
posteam                      119
defteam                      119
fourth_down_converted        119
fourth_down_failed           119
drive                          0
side_of_field                  0
yardline_100                 119
down                           0
ydstogo                        0
play_type                    119
yards_gained                   4
touchdown                    119
td_team                    39020
field_goal_attempt           119
field_goal_result          30625
kick_distance               8492
punt_attempt                 119
incomplete_pass              119
interception                 119
home_timeouts_remaining        0
away_timeouts_remaining        0
td_prob   

In [9]:
# addreessing null valuea in core features by identifying the rows with clear possession team, no outcome and contains outcome contradictions 
# start with rows where no clear possession team is identified

mask_valid_teams = (
    fourth_down_scenarios['posteam'].notna() &
    fourth_down_scenarios['defteam'].notna()
)

fourth_down_scenarios = fourth_down_scenarios[mask_valid_teams].copy()



In [10]:
# save this interim dataset under interim folder 

fourth_down_scenarios.to_csv(
    "../../data/interim/fourth_down_scenarios.csv",
    index=False
)

In [11]:
# check for duplicates 

dupes = fourth_down_scenarios.duplicated(subset=['game_id', 'play_id'], keep=False)
fourth_down_scenarios[dupes].sort_values(['game_id', 'play_id']).head(20)


,game_id,play_id,home_team,away_team,qtr,game_seconds_remaining,posteam_score,defteam_score,score_differential,posteam,...,kick_distance,punt_attempt,incomplete_pass,interception,home_timeouts_remaining,away_timeouts_remaining,td_prob,fg_prob,ep,epa
431273,2018110800,202,PIT,CAR,1,3379.0,0.0,0.0,0.0,CAR,...,NaN,0.0,0.0,0.0,3,3,0.21,0.62,2.46,1.96
433662,2018110800,202,PIT,CAR,1,3379.0,0.0,0.0,0.0,CAR,...,NaN,0.0,0.0,0.0,3,3,0.21,0.62,2.46,1.96
431286,2018110800,491,PIT,CAR,1,3246.0,7.0,13.0,-6.0,CAR,...,45.0,1.0,0.0,0.0,3,2,0.19,0.10,-2.19,0.01
433675,2018110800,491,PIT,CAR,1,3246.0,7.0,13.0,-6.0,CAR,...,45.0,1.0,0.0,0.0,3,2,0.19,0.10,-2.19,0.01
431307,2018110800,1001,PIT,CAR,2,2696.0,7.0,20.0,-13.0,CAR,...,33.0,1.0,0.0,0.0,3,2,0.17,0.47,0.89,-0.34
433696,2018110800,1001,PIT,CAR,2,2696.0,7.0,20.0,-13.0,CAR,...,33.0,1.0,0.0,0.0,3,2,0.17,0.47,0.89,-0.34
431316,2018110800,1212,PIT,CAR,2,2391.0,20.0,7.0,13.0,PIT,...,50.0,0.0,0.0,0.0,3,2,0.04,0.72,1.36,1.64
433705,2018110800,1212,PIT,CAR,2,2391.0,20.0,7.0,13.0,PIT,...,50.0,0.0,0.0,0.0,3,2,0.04,0.72,1.36,1.64
431347,2018110800,1913,PIT,CAR,2,1869.0,14.0,30.0,-16.0,CAR,...,46.0,1.0,0.0,0.0,1,1,0.06,0.15,0.36,-0.71
433736,2018110800,1913,PIT,CAR,2,1869.0,14.0,30.0,-16.0,CAR,...,46.0,1.0,0.0,0.0,1,1,0.06,0.15,0.36,-0.71


 ## 20 rows matched exactly 
 fourth_down_scenarios = (
    fourth_down_scenarios
    .drop_duplicates(subset=['game_id', 'play_id'])
    .copy()
)


In [12]:
fourth_down_scenarios = (
    fourth_down_scenarios
    .drop_duplicates(subset=['game_id', 'play_id'])
    .copy()
)


In [13]:

print(fourth_down_scenarios.shape)
print()
print(fourth_down_scenarios.info())

(39336, 34)

<class 'pandas.core.frame.DataFrame'>
Index: 39336 entries, 4 to 449368
Data columns (total 34 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   game_id                  39336 non-null  int64  
 1   play_id                  39336 non-null  int64  
 2   home_team                39336 non-null  object 
 3   away_team                39336 non-null  object 
 4   qtr                      39336 non-null  int64  
 5   game_seconds_remaining   39333 non-null  float64
 6   posteam_score            39336 non-null  float64
 7   defteam_score            39336 non-null  float64
 8   score_differential       39336 non-null  float64
 9   posteam                  39336 non-null  object 
 10  defteam                  39336 non-null  object 
 11  fourth_down_converted    39336 non-null  float64
 12  fourth_down_failed       39336 non-null  float64
 13  drive                    39336 non-null  int64  
 14  side_of_field

In [14]:
# normalize / format text columns for easy reading and modeling

fourth_down_scenarios['play_type_clean'] = (
    fourth_down_scenarios['play_type']
    .str.strip()
    .str.lower()
)

fourth_down_scenarios['field_goal_result_clean'] = (
    fourth_down_scenarios['field_goal_result']
    .str.strip()
    .str.lower()
)